In [1]:
import pandas as pd
data=pd.read_csv(r"phishingemails.csv")

In [2]:
data.head()

,Label,cleaned_text,final_processed_text
0,Safe Email,re disc uniformitarianism re sex la...,r e d i s c u n i f o r m i t ...
1,Safe Email,the other side of galicismos galicismo is ...,t h e o t h e r s i d e o f g a l i ...
2,Safe Email,re equistar deal tickets are you still availa...,r e e q u i s t a r d e a l t i c k e ...
3,Phishing Email,\nhello i am your hot lil horny toy\n i am ...,\n h e l l o i a m y o u r h o t l i...
4,Phishing Email,software at incredibly low prices lower d...,s o f t w a r e a t i n c r e d i b l y ...


In [3]:
data = data.drop('cleaned_text', axis=1)
display(data.head())

,Label,final_processed_text
0,Safe Email,r e d i s c u n i f o r m i t ...
1,Safe Email,t h e o t h e r s i d e o f g a l i ...
2,Safe Email,r e e q u i s t a r d e a l t i c k e ...
3,Phishing Email,\n h e l l o i a m y o u r h o t l i...
4,Phishing Email,s o f t w a r e a t i n c r e d i b l y ...


In [4]:
data.head()

,Label,final_processed_text
0,Safe Email,r e d i s c u n i f o r m i t ...
1,Safe Email,t h e o t h e r s i d e o f g a l i ...
2,Safe Email,r e e q u i s t a r d e a l t i c k e ...
3,Phishing Email,\n h e l l o i a m y o u r h o t l i...
4,Phishing Email,s o f t w a r e a t i n c r e d i b l y ...


In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np



In [12]:
# --- 1. Load and Prepare Data ---
# Load the data from your CSV file.
try:
    df = pd.read_csv("phishingemails.csv")
except FileNotFoundError:
    print("Error: 'phishingemails.csv' not found. Please ensure the file is in the same directory as the script.")
    exit()



In [13]:
# It's good practice to see what the data looks like.
print("--- Initial Dataset ---")
print(df.head())
print("\n" + "="*30 + "\n")

# --- 2. Preprocess and Clean Data ---
# Drop rows where the email text is missing, as they can't be processed.
df.dropna(subset=['final_processed_text'], inplace=True)


--- Initial Dataset ---
            Label                                       cleaned_text  \
0      Safe Email  re      disc  uniformitarianism  re    sex  la...   
1      Safe Email  the other side of  galicismos   galicismo  is ...   
2      Safe Email  re  equistar deal tickets are you still availa...   
3  Phishing Email  \nhello i am your hot lil horny toy\n    i am ...   
4  Phishing Email  software at incredibly low prices    lower   d...   

                                final_processed_text  
0  r e             d i s c     u n i f o r m i t ...  
1  t h e   o t h e r   s i d e   o f     g a l i ...  
2  r e     e q u i s t a r   d e a l   t i c k e ...  
3  \n h e l l o   i   a m   y o u r   h o t   l i...  
4  s o f t w a r e   a t   i n c r e d i b l y   ...  




In [14]:

# Isolation Forest identifies outliers. We'll label phishing emails as anomalies (-1)
# and safe emails as normal (1).
df['Label_encoded'] = df['Label'].apply(lambda x: -1 if x == 'Phishing Email' else 1)

# Separate features (X) and the ground truth labels (y)
X_text = df['final_processed_text']
y_true = df['Label_encoded']


In [15]:
# --- 3. Feature Extraction (Text to Numbers) ---
# We need to convert the email text into numerical vectors.
# FIX: Because the text is formatted with spaces between letters (e.g., 'h e l l o'),
# we use analyzer='char' to treat sequences of characters as features.
# The default 'word' analyzer would fail as it would see single letters as stop words.
print("--- Vectorizing Text Data ---")
vectorizer = TfidfVectorizer(
    max_features=1000,
    analyzer='char',  # Analyze at the character level
    ngram_range=(2, 5)  # Look for character sequences of length 2 to 5 to find patterns
)
X_tfidf = vectorizer.fit_transform(X_text)
print(f"Data transformed into a matrix of shape: {X_tfidf.shape}")
print("\n" + "="*30 + "\n")




--- Vectorizing Text Data ---
Data transformed into a matrix of shape: (18634, 1000)




In [16]:
# --- 4. Train the Isolation Forest Model ---
# The 'contamination' parameter tells the model what proportion of the
# dataset is expected to be anomalous (phishing). We calculate this from our data.
contamination_rate = df[df['Label'] == 'Phishing Email'].shape[0] / df.shape[0]

# If contamination_rate is 0, it can cause issues. Set a small default if so.
if contamination_rate == 0:
    contamination_rate = 'auto'

print(f"--- Training Isolation Forest Model ---")
print(f"Calculated contamination rate: {contamination_rate:.2f}")

# Initialize and train the model
# random_state is set for reproducibility
iso_forest = IsolationForest(n_estimators=100, contamination=contamination_rate, random_state=42)
iso_forest.fit(X_tfidf)




--- Training Isolation Forest Model ---
Calculated contamination rate: 0.39


IsolationForest(contamination=0.3924009874423098, random_state=42)

In [17]:
# --- 5. Make Predictions ---
# Predict on the same data to see how well it learned to separate the classes.
y_pred = iso_forest.predict(X_tfidf)

print("Model training complete. Predictions are made.")
print("\n" + "="*30 + "\n")


# --- 6. Evaluate the Model ---
print("--- Model Evaluation ---")



Model training complete. Predictions are made.


--- Model Evaluation ---


In [18]:
# Define target names for the report
target_names = ['Phishing Email', 'Safe Email']

# Print the classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=target_names, labels=[-1, 1]))

# Print the confusion matrix
print("Confusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=[-1, 1])
cm_df = pd.DataFrame(cm, index=['True: Phishing', 'True: Safe'], columns=['Pred: Phishing', 'Pred: Safe'])
print(cm_df)
print("\n")

# Calculate and print accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Overall Accuracy: {accuracy:.2%}")


Classification Report:
                precision    recall  f1-score   support

Phishing Email       0.44      0.44      0.44      7312
    Safe Email       0.64      0.64      0.64     11322

      accuracy                           0.56     18634
     macro avg       0.54      0.54      0.54     18634
  weighted avg       0.56      0.56      0.56     18634

Confusion Matrix:
                Pred: Phishing  Pred: Safe
True: Phishing            3235        4077
True: Safe                4077        7245


Overall Accuracy: 56.24%


In [22]:
model_filename = 'isolation_forest_model.pkl'
vectorizer_filename = 'tfidf_vectorizer.pkl'

In [25]:
import pickle

with open(model_filename, 'wb') as file:
    pickle.dump(iso_forest, file)
print(f"Model saved to '{model_filename}'")

# Save the fitted vectorizer to a file
with open(vectorizer_filename, 'wb') as file:
    pickle.dump(vectorizer, file)
print(f"Vectorizer saved to '{vectorizer_filename}'")

Model saved to 'isolation_forest_model.pkl'
Vectorizer saved to 'tfidf_vectorizer.pkl'


In [24]:
!pip install pickle

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


In [29]:
import pickle
import numpy as np

# --- 1. Load the Saved Model and Vectorizer ---
# Define the filenames. These must match the files saved from the training script.
model_filename = 'isolation_forest_model.pkl'
vectorizer_filename = 'tfidf_vectorizer.pkl'

# Load the model and vectorizer from the .pkl files
try:
    with open(model_filename, 'rb') as file:
        loaded_model = pickle.load(file)
    print(f"Model loaded successfully from '{model_filename}'")

    with open(vectorizer_filename, 'rb') as file:
        loaded_vectorizer = pickle.load(file)
    print(f"Vectorizer loaded successfully from '{vectorizer_filename}'")
except FileNotFoundError:
    print("Error: Model or vectorizer file not found.")
    print("Please make sure 'isolation_forest_model.pkl' and 'tfidf_vectorizer.pkl' are in the same directory.")
    exit()




Model loaded successfully from 'isolation_forest_model.pkl'
Vectorizer loaded successfully from 'tfidf_vectorizer.pkl'


In [30]:
# --- 2. Create a Prediction Function ---
def classify_email(email_text):
    """
    Classifies a single email text as 'Phishing Email' or 'Safe Email'.

    Args:
        email_text (str): The raw text content of the email to classify.

    Returns:
        str: The prediction, either 'Phishing Email' or 'Safe Email'.
    """
    # The vectorizer expects a list of documents, so we put the single email text into a list.
    text_features = loaded_vectorizer.transform([email_text])

    # Use the loaded model to make a prediction.
    # The output will be -1 for an anomaly (phishing) and 1 for normal (safe).
    prediction = loaded_model.predict(text_features)

    # Return the corresponding label
    if prediction[0] == -1:
        return "Phishing Email"
    else:
        return "Safe Email"



In [38]:
# --- 3. Test with Your Own Input ---
if __name__ == "__main__":
    # Example 1: A suspicious-looking email
    suspicious_email = "urgent action required your account has been compromised click here to update your password immediately"
    prediction1 = classify_email(suspicious_email)
    print(f"\nInput: '{suspicious_email}'")
    print(f"Prediction: {prediction1}")

    print("-" * 30)

    # Example 2: A normal-looking email
    safe_email = "Hi team, just a reminder about our meeting tomorrow at 10am. Please review the attached agenda."
    prediction2 = classify_email(safe_email)
    print(f"Input: '{safe_email}'")
    print(f"Prediction: {prediction2}")

    print("-" * 30)

    # --- Test with your own custom input ---
    your_input = input("Enter your own email text to classify: ")
    your_prediction = classify_email(your_input)
    print(f"Prediction for your input: {your_prediction}")




Input: 'urgent action required your account has been compromised click here to update your password immediately'
Prediction: Safe Email
------------------------------
Input: 'Hi team, just a reminder about our meeting tomorrow at 10am. Please review the attached agenda.'
Prediction: Safe Email
------------------------------
Enter your own email text to classify: Subject: IRS Alert: Unclaimed Tax Refund of $642.80 From: "IRS Tax Department" notice@irs-gov.org  Body: You have an unclaimed tax refund from last year. Failure to claim within 72 hours will void it.  📥 Download your refund form here: [IRS_Form_2024.zip]   IRS Processing Center 
Prediction for your input: Phishing Email
